# Generative AI — Assignment 1

**Part 1:** Topic Detection and Summarization of News Articles (45 marks)
**Part 2:** Job Postings Analysis — Role Categorization and Requirements Extraction (55 marks)

Everything below is built with **LangChain** — prompt templates, LCEL chains, output parsers and
structured output — on top of `openai/gpt-oss-120b` served by **Groq**, the same stack used in class.

| Step | What it does | Where |
|---|---|---|
| Setup | Load the API key, initialise the rate-limited chat model | Section 0 |
| P1.1–P1.5 | Load news → classify topic → summarise → extract entities → merged DataFrame | Part 1 |
| P2.1–P2.5 | Load jobs → classify domain → extract skills/education/experience → merged DataFrame | Part 2 |

## 0. Setup

The Groq API key is read from the `.env` file at the repository root (as in the class notebooks),
with a prompt as a fallback.

Two Groq models are used, each with its own rate budget: `gpt-oss-120b` for the structured extraction steps and `gpt-oss-20b` for the lighter text tasks (topic labels and summaries). This matches the brief's advice to balance inference speed against accuracy for each use case.

In [1]:
import json, re, time
from typing import List

import pandas as pd

In [2]:
import os, getpass
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True), override=True)

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("GROQ API Key: ")

print(f"Groq API key loaded, begins with: {os.environ['GROQ_API_KEY'][:4]}...")

Groq API key loaded, begins with: gsk_...


In [3]:
from langchain.chat_models import init_chat_model
from langchain_core.rate_limiters import InMemoryRateLimiter

MODEL_NAME = "openai/gpt-oss-120b"      # extraction: entities, job requirements (structured output)
FAST_MODEL_NAME = "openai/gpt-oss-20b"  # light text tasks: topic/domain labels, summaries

# Groq's free tier caps *each model* at 8,000 tokens per minute. Splitting the work across two models
# gives the notebook two independent budgets, roughly halving the run time.
#
# Why the split falls where it does: gpt-oss-20b is fast and handles plain-text answers well, but it
# is unreliable at structured output (it sometimes returns no tool call at all), so every chain that
# returns a Pydantic schema stays on gpt-oss-120b.
#
# Each model gets a rate limiter that spaces its requests evenly. Groq charges each request against the
# budget at an estimate of roughly twice its real token use, so the practical ceiling is ~7 requests a
# minute per model. Relying on retries alone was tried and failed: parallel calls hit the cap together,
# are all told to retry in ~0.2s, collide again, and run out of retries.
llm = init_chat_model(
    MODEL_NAME,
    model_provider="groq",
    temperature=0.0,  # deterministic, reproducible classification and extraction
    rate_limiter=InMemoryRateLimiter(requests_per_second=0.13, check_every_n_seconds=0.2, max_bucket_size=1),
    max_retries=10,   # absorbs the occasional 429 - the client waits out the retry-after header
)

llm_fast = init_chat_model(
    FAST_MODEL_NAME,
    model_provider="groq",
    temperature=0.0,
    rate_limiter=InMemoryRateLimiter(requests_per_second=0.21, check_every_n_seconds=0.2, max_bucket_size=1),
    max_retries=10,
)

In [4]:
# Quick connectivity check before spending time on the full dataset.
print("Model replied:", llm.invoke("Reply with the single word OK.").content)

Model replied: OK


In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_core.utils.function_calling import convert_to_json_schema
from langchain_core.runnables import RunnablePassthrough
from pydantic import BaseModel, Field

# Every article / description is truncated before it goes to the model, which keeps each call small
# against the tokens-per-minute cap; the median article is ~1,900 characters, so most go in whole.
MAX_CHARS = 2000

# How many rows each part processes. The assignment asks for 30 news articles and 25 job postings;
# raise these to run the optional bonus over the full datasets.
N_ARTICLES = 30
N_JOBS = 25

# .batch() keeps a few requests in flight; the rate limiter above still decides the actual pace.
MAX_CONCURRENCY = 6   # enough rows in flight to keep both models' budgets busy

# Topic classification only reads the headline plus the lead paragraph: news copy front-loads
# what a story is about, and it cuts the classification call to about half the tokens.
LEAD_CHARS = 800


def structured_output(model, schema):
    """Return a runnable that makes `model` answer as a `schema` instance.

    Uses Groq's *strict* JSON-schema mode: the server constrains generation to the schema, so the reply
    is always valid JSON of the right shape. The default `with_structured_output` (tool calling) is kept
    as a fallback. Tool calling alone is not enough here: on some articles gpt-oss answers with a
    Markdown list instead of calling the tool, Groq rejects it ("tool_use_failed"), and because
    temperature is 0 every retry fails the same way.
    """
    json_schema = convert_to_json_schema(schema)
    json_schema["additionalProperties"] = False  # required by strict mode
    strict = model.bind(response_format={
        "type": "json_schema",
        "json_schema": {"name": json_schema["title"], "strict": True, "schema": json_schema},
    }) | PydanticOutputParser(pydantic_object=schema)
    return strict.with_fallbacks([model.with_structured_output(schema)])
RETRY_ATTEMPTS = 3

---
# Part 1: Topic Detection and Summarization of News Articles (45 marks)

## Step 1: Load the Dataset

The BBC News Full-Text archive: 2,225 articles labelled with one of 5 categories. The file is
**tab-separated**, so `sep="\t"` is required.

In [6]:
news_raw = pd.read_csv("bbc-news-data.csv", sep="\t")

print("Shape:", news_raw.shape)
print("\nArticles per category:")
print(news_raw["category"].value_counts())
news_raw.head()

Shape: (2225, 4)

Articles per category:
category
sport            511
business         510
politics         417
tech             401
entertainment    386
Name: count, dtype: int64


,category,filename,title,content
0,business,001.txt,Ad sales boost Time Warner profit,Quarterly profits at US media giant TimeWarne...
1,business,002.txt,Dollar gains on Greenspan speech,The dollar has hit its highest level against ...
2,business,003.txt,Yukos unit buyer faces loan claim,The owners of embattled Russian oil giant Yuk...
3,business,004.txt,High fuel prices hit BA's profits,British Airways has blamed high fuel prices f...
4,business,005.txt,Pernod takeover talk lifts Domecq,Shares in UK drinks and food firm Allied Dome...


In [7]:
# Limit the data to the first 30 articles to keep processing time manageable.
df_news = news_raw.head(N_ARTICLES).copy().reset_index(drop=True)
df_news.insert(0, "Article_ID", df_news.index)

# The text actually sent to the model - the title gives it a useful headline signal.
df_news["Article_Text"] = (
    df_news["title"].str.strip() + "\n\n" + df_news["content"].str.strip()
).str.slice(0, MAX_CHARS)

print("Working set:", df_news.shape)
df_news[["Article_ID", "category", "title"]].head()

Working set: (30, 6)


,Article_ID,category,title
0,0,business,Ad sales boost Time Warner profit
1,1,business,Dollar gains on Greenspan speech
2,2,business,Yukos unit buyer faces loan claim
3,3,business,High fuel prices hit BA's profits
4,4,business,Pernod takeover talk lifts Domecq


## Step 2: Define the Topic Classification Task (10 marks)

A `ChatPromptTemplate` carrying a system instruction, a one-line gloss of each category, and
**three few-shot examples** (article snippet → correct label). The few-shot pairs are what stop a
small model from answering in a full sentence.

A normaliser runs after the parser and maps the answer onto one of the five allowed labels. If an
answer contains no valid label at all (this happened once, on a one-off empty reply), the normaliser
raises and the chain's retry asks again, so an invalid label can never reach the DataFrame.

Classification is the lightest task in Part 1, so it runs on `gpt-oss-20b` and reads only the headline plus the first ~800 characters — the lead paragraph, where a news story states what it is about.

In [8]:
VALID_TOPICS = ["Business", "Entertainment", "Politics", "Sport", "Tech"]

CATEGORY_GLOSS = """- Business: companies, markets, the economy, earnings, jobs, trade
- Entertainment: film, music, television, celebrities, books, art and awards
- Politics: government, ministers, elections, parties, laws and public policy
- Sport: matches, athletes, clubs, tournaments and results
- Tech: technology itself - the internet, computing, gadgets, telecoms, video games, digital media"""

topic_prompt = ChatPromptTemplate([
    ("system",
     "You are a news desk editor for the BBC archive. Analyze the following news article and "
     "identify its topic as one of the following categories:\n"
     f"{CATEGORY_GLOSS}\n\n"
     "If an article is mainly about a technology, a device or the internet, choose Tech even when "
     "the setting is political or commercial.\n"
     "Answer with the category label only - no explanation, no punctuation, no other words."),
    # Few-shot example 1
    ("human", "Article: Shares in the airline fell 4% on Tuesday after it cut its full-year profit "
              "forecast, blaming higher fuel costs and weaker transatlantic demand."),
    ("ai", "Business"),
    # Few-shot example 2
    ("human", "Article: The defending champions came from two sets down to win the semi-final in "
              "four hours, setting up a title clash on Sunday."),
    ("ai", "Sport"),
    # Few-shot example 3
    ("human", "Article: Officials say the new electronic voting terminals cut queues at polling "
              "stations, though campaigners question how the machines store ballots."),
    ("ai", "Tech"),
    # The actual article
    ("human", "Article: {article}"),
])


def normalise_topic(raw: str) -> str:
    """Map the model's answer onto one of the five allowed labels.

    If the answer contains no valid label (rare - e.g. an empty reply), raise instead of guessing:
    the .with_retry() below then asks the model again, rather than writing a bad label to the column.
    """
    cleaned = raw.strip().strip(".").lower()
    for topic in VALID_TOPICS:
        if topic.lower() in cleaned:
            return topic
    raise ValueError(f"No valid topic label in the model's answer: {raw!r}")


def lead_only(inputs: dict) -> dict:
    """Keep the headline and lead paragraph - all the classifier needs."""
    return {"article": inputs["article"][:LEAD_CHARS]}


# LCEL: trim to the lead -> prompt -> model -> string parser -> normaliser
# If gpt-oss-20b still gives no valid label after its retries, gpt-oss-120b classifies the full text.
topic_chain = (lead_only | topic_prompt | llm_fast | StrOutputParser() | normalise_topic).with_retry(
    stop_after_attempt=RETRY_ATTEMPTS
).with_fallbacks([topic_prompt | llm | StrOutputParser() | normalise_topic])

In [9]:
# Expected output: show this works for a sample datapoint.
sample = df_news.iloc[0]

sample_topic = topic_chain.invoke({"article": sample["Article_Text"]})

print("TITLE          :", sample["title"])
print("TRUE CATEGORY  :", sample["category"])
print("DETECTED TOPIC :", sample_topic)

TITLE          : Ad sales boost Time Warner profit
TRUE CATEGORY  : business
DETECTED TOPIC : Business


## Step 3: Define the Summarization Task (10 marks)

A second chain over the same article, asking for 2–3 factual sentences covering
who / what / when / where / why, with no personal commentary. Small models like to open with
*"Here is a summary of the article:"*, so a tiny post-processing step strips that preamble — the
instruction handles most cases and the cleaner catches the rest.

In [10]:
summary_prompt = ChatPromptTemplate([
    ("system",
     "You are a news summarizer. Summarize the main points of the news article in 2-3 sentences.\n"
     "Cover who, what, when, where and why as far as the article states them.\n"
     "Report only what the article says - no opinions and no commentary.\n"
     "Start directly with the first sentence of the summary: do not write any preamble such as "
     "'Here is a summary', and do not use bullet points."),
    ("human", "{article}"),
])


def strip_preamble(text: str) -> str:
    """Drop a leading 'Here is a summary:'-style lead-in if the model adds one anyway."""
    return re.sub(r"^(here (is|are)|sure|below is)[^\n:]*:\s*", "", text.strip(), flags=re.I).strip()


summary_chain = (summary_prompt | llm_fast | StrOutputParser() | strip_preamble).with_retry(
    stop_after_attempt=RETRY_ATTEMPTS
)

In [11]:
# Expected output: show this works for a sample datapoint.
sample_summary = summary_chain.invoke({"article": sample["Article_Text"]})

print("TITLE  :", sample["title"], "\n")
print("SUMMARY:", sample_summary)

TITLE  : Ad sales boost Time Warner profit 

SUMMARY: Time Warner reported a 76 % jump in quarterly profit to $1.13 bn for the three months to December 2004, driven by higher high‑speed internet sales, increased advertising revenue and one‑off gains that offset a dip at Warner Bros. The U.S. media giant, which now owns 8 % of Google, also noted a 2 % rise in fourth‑quarter sales to $11.1 bn, though its AOL unit lost 464,000 subscribers despite an 8 % rise in underlying profit before exceptional items. For the full year, Time Warner posted a 27 % profit increase to $3.36 bn on revenue up 6.4 % to $42.09 bn, and it is projecting about 5 % operating earnings growth for 2005 while preparing to restate its 2000 and 2003 accounts.


## Step 4: Key Entity Extraction (10 marks)

Entity extraction is **chained after summarization** — the prompt receives both the summary (which
says what matters) and the article (which holds the exact names).

Rather than parsing a comma-separated string by hand, the model answers against a Pydantic schema using Groq's **strict JSON-schema mode** (the `structured_output` helper in Section 0), with `with_structured_output(...)` as a fallback. Strict mode matters: with plain tool calling, gpt-oss occasionally replies with a Markdown list instead of calling the tool, and at temperature 0 every retry repeats the same failure.

One subtlety worth recording: every field in the schema is **required**. Giving a field a default
makes it optional in the generated schema, and a model is then free to omit it — Pydantic fills the
default and the column silently ends up empty. (This was observed in practice while building this
notebook: with defaulted fields, a smaller model returned empty lists on every single row.)
Required fields plus an explicit "use [] if none" in the description avoid that failure mode.

In [12]:
class KeyEntities(BaseModel):
    """Important named entities mentioned in a news article."""

    # None of these fields carries a default: a defaulted field is *optional* in the schema, so the
    # model may omit it and Pydantic silently fills the default. Required fields force the model to
    # produce each list; "use []" lives in the description instead.
    people: List[str] = Field(
        description="Every notable person named in the article, e.g. ['Alan Greenspan']. "
                    "Use [] if the article names none.",
    )
    organizations: List[str] = Field(
        description="Every company, team, government or institution named, e.g. ['Time Warner']. "
                    "Use [] if the article names none.",
    )
    locations: List[str] = Field(
        description="Every city, country or region named, e.g. ['New York']. "
                    "Use [] if the article names none.",
    )


entity_prompt = ChatPromptTemplate([
    ("system",
     "From the article below, list the names of any important people, organizations and places "
     "mentioned.\n"
     "Only list entities that literally appear in the article - never invent one. "
     "Use each entity's name as written, and do not repeat the same entity twice."),
    ("human", "Summary of the article:\n{Summary}\n\nFull article:\n{article}"),
])

entity_chain = (entity_prompt | structured_output(llm, KeyEntities)).with_retry(
    stop_after_attempt=RETRY_ATTEMPTS
)

In [13]:
# Expected output: show this works for a sample datapoint (chained onto the Step 3 summary).
sample_entities = entity_chain.invoke({"article": sample["Article_Text"], "Summary": sample_summary})

print("TITLE:", sample["title"], "\n")
print("People       :", sample_entities.people)
print("Organizations:", sample_entities.organizations)
print("Locations    :", sample_entities.locations)

TITLE: Ad sales boost Time Warner profit 

People       : ['Richard Parsons']
Organizations: ['TimeWarner', 'Time Warner', 'Google', 'Warner Bros', 'AOL', 'U.S. Securities Exchange Commission (SEC)', 'SEC']
Locations    : ['U.S.', 'US']


## Step 5: Update the DataFrame with Results (15 marks)

The three chains are composed into a single runnable with `RunnablePassthrough.assign`:

* `Detected_Topic` and `Summary` are produced **in parallel** (each only needs the article),
* `Key_Entities` then runs on that result, so it sees the summary produced a moment earlier.

In [14]:
article_chain = (
    RunnablePassthrough.assign(Detected_Topic=topic_chain, Summary=summary_chain)
    | RunnablePassthrough.assign(Key_Entities=entity_chain)
)

article_chain.get_graph().print_ascii()

                      +---------------------------------------+                      
                      | Parallel<Detected_Topic,Summary>Input |                      
                      +---------------------------------------+                      
                        ******             *            ******                       
                   *****                    *                 *****                  
                ***                         *                      *****             
+--------------------+                      *                           ***          
| ChatPromptTemplate |                      *                             *          
+--------------------+                      *                             *          
           *                                *                             *          
           *                                *                             *          
           *                                *         

In [15]:
# Iterate over every article. .batch() is the same loop as a for-loop over the rows, but it keeps
# MAX_CONCURRENCY requests in flight. Topic and summary run on gpt-oss-20b while entities
# run on gpt-oss-120b, so the two rate budgets are consumed in parallel.
start = time.time()

article_inputs = [{"article": text} for text in df_news["Article_Text"]]
article_results = article_chain.batch(article_inputs, config={"max_concurrency": MAX_CONCURRENCY})

print(f"Processed {len(article_results)} articles in {time.time() - start:.1f}s")

Processed 30 articles in 512.1s


In [16]:
# Populate the new columns from the model's outputs.
df_news["Detected_Topic"] = [r["Detected_Topic"] for r in article_results]
df_news["Summary"] = [r["Summary"] for r in article_results]

# Key_Entities as a single flat list (as in the assignment example), plus the breakdown by type.
df_news["Key_Entities"] = [
    r["Key_Entities"].people + r["Key_Entities"].organizations + r["Key_Entities"].locations
    for r in article_results
]
df_news["People"] = [r["Key_Entities"].people for r in article_results]
df_news["Organizations"] = [r["Key_Entities"].organizations for r in article_results]
df_news["Locations"] = [r["Key_Entities"].locations for r in article_results]

print("Columns now:", list(df_news.columns))

Columns now: ['Article_ID', 'category', 'filename', 'title', 'content', 'Article_Text', 'Detected_Topic', 'Summary', 'Key_Entities', 'People', 'Organizations', 'Locations']


In [17]:
# Final Pandas DataFrame: all original columns plus the new ones.
pd.set_option("display.max_colwidth", 90)

df_news_final = df_news[
    ["Article_ID", "category", "filename", "title", "content",
     "Detected_Topic", "Summary", "Key_Entities", "People", "Organizations", "Locations"]
]

df_news_final[["Article_ID", "title", "Detected_Topic", "Summary", "Key_Entities"]]

,Article_ID,title,Detected_Topic,Summary,Key_Entities
0,0,Ad sales boost Time Warner profit,Business,Time Warner’s quarterly profit for the three months to December 2004 rose 76% to $1.13...,"[Richard Parsons, TimeWarner, Time Warner, Google, AOL, Warner Bros, U.S. Securities E..."
1,1,Dollar gains on Greenspan speech,Business,The U.S. dollar rose to its highest level against the euro in almost three months afte...,"[Alan Greenspan, Robert Sinche, Federal Reserve, Bank of America, G7, London, New York..."
2,2,Yukos unit buyer faces loan claim,Business,"Menatep Group, the owners of the former Russian oil giant Yukos, will demand that Rosn...","[Mikhail Khodorkovsky, Jamie Firestone, Tim Osborne, Rosneft, Menatep Group, Yukos, Re..."
3,3,High fuel prices hit BA's profits,Business,British Airways reported a 40 % drop in pre‑tax profit for the quarter ended 31 Decemb...,"[Rod Eddington, Mike Powell, Martin Broughton, Nick Van den Brul, British Airways, BA,..."
4,4,Pernod takeover talk lifts Domecq,Business,Shares of Allied Domecq rose 4% in London by 1200 GMT after reports that France’s Pern...,"[Allied Domecq, Pernod Ricard, Pernod, Wall Street Journal, Financial Times, Seagram, ..."
5,5,Japan narrowly escapes recession,Business,Japan’s economy narrowly avoided a technical recession in the three months to Septembe...,"[Heizo Takenaka, Paul Sheard, Lehman Brothers, Japan, Tokyo]"
6,6,Jobs growth still slow in the US,Business,"In January, U.S. firms added 146,000 jobs—below the 190,000 expected—yet the unemploym...","[President Bush, Herbert Hoover, Rick Egelton, Ken Mayland, Labor Department, BMO Fina..."
7,7,India calls for fair trade rules,Politics,"India’s finance minister Palaniappan Chidambaram, attending the G7 meeting in London o...","[Palaniappan Chidambaram, Gordon Brown, G7, G20, United Nations, World Bank, IMF, Lond..."
8,8,Ethiopia's crop production up 24%,Business,"Ethiopia produced 14.27 million tonnes of crops in 2004, a 24 % rise over 2003 and 21 ...","[Henri Josserand, Food and Agriculture Organisation, World Food Programme, FAO, FAO's ..."
9,9,Court rejects $280bn tobacco case,Politics,An appeal court in Washington rejected a $280 billion claim filed by the Clinton admin...,"[Clinton administration, Altria Group, RJ Reynolds Tobacco, Lorillard Tobacco, Liggett..."


In [18]:
# The same record in the JSON shape asked for in the assignment.
row = df_news_final.iloc[0]

print(json.dumps({
    "Article_ID": int(row["Article_ID"]),
    "Title": row["title"],
    "Article_Text": row["content"][:200].strip() + "... [excerpt]",
    "Detected_Topic": row["Detected_Topic"],
    "Summary": row["Summary"],
    "Key_Entities": row["Key_Entities"],
}, indent=2))

{
  "Article_ID": 0,
  "Title": "Ad sales boost Time Warner profit",
  "Article_Text": "Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (\u00a3600m) for the three months to December, from $639m year-earlier.  The firm, which is now one of the biggest investors in Google,... [excerpt]",
  "Detected_Topic": "Business",
  "Summary": "Time Warner\u2019s quarterly profit for the three months to December 2004 rose 76% to $1.13\u202fbillion, driven by higher high\u2011speed internet sales, increased advertising revenue and one\u2011off gains that offset a dip at Warner Bros and a loss of 464,000 AOL subscribers. The U.S. media giant, which now owns 8\u202f% of Google, also reported that its underlying AOL profit before exceptional items grew 8\u202f% and that it plans to offer the service free to Time Warner internet customers to boost subscriptions. For the full year 2004, Time Warner posted a $3.36\u202fbillion profit (up 27% from 2003) on revenues of $42.09\u202fbillio

In [19]:
df_news_final.to_csv("part1_news_analysis_output.csv", index=False)
print("Saved -> part1_news_analysis_output.csv")

Saved -> part1_news_analysis_output.csv


### Sanity check: detected topic vs. the dataset's own label

The BBC dataset ships with a ground-truth `category`, so Step 2 can be scored rather than just
eyeballed. One caveat: the file is sorted by category, so `head(30)` is 30 *business* articles —
accuracy on those 30 only tests one class. A second check therefore runs the same chain over a
balanced sample of 2 articles from each of the 5 categories.

In [20]:
truth = df_news_final["category"].str.lower()
pred = df_news_final["Detected_Topic"].str.lower()

print(f"Accuracy on the {len(df_news_final)} processed articles: {(truth == pred).mean():.1%}")
print("(all of them are 'business' - see the caveat above)\n")

mismatches = df_news_final.loc[truth != pred, ["Article_ID", "title", "category", "Detected_Topic"]]
print(f"Mismatches: {len(mismatches)}")
mismatches

Accuracy on the 30 processed articles: 80.0%
(all of them are 'business' - see the caveat above)

Mismatches: 6


,Article_ID,title,category,Detected_Topic
7,7,India calls for fair trade rules,business,Politics
9,9,Court rejects $280bn tobacco case,business,Politics
10,10,Ask Jeeves tips online ad revival,business,Tech
11,11,Indonesians face fuel price rise,business,Politics
14,14,Air passengers win new EU rights,business,Politics
18,18,India widens access to telecoms,business,Tech


In [21]:
# Balanced check: 2 articles from each category, classified with the same Step 2 chain.
balanced = (news_raw.groupby("category", group_keys=False).head(2)).reset_index(drop=True)
balanced["Article_Text"] = (
    balanced["title"].str.strip() + "\n\n" + balanced["content"].str.strip()
).str.slice(0, MAX_CHARS)

balanced["Detected_Topic"] = topic_chain.batch(
    [{"article": text} for text in balanced["Article_Text"]],
    config={"max_concurrency": MAX_CONCURRENCY},
)

balanced_accuracy = (
    balanced["category"].str.lower() == balanced["Detected_Topic"].str.lower()
).mean()
print(f"Balanced-sample accuracy ({len(balanced)} articles, 2 per category): {balanced_accuracy:.0%}\n")

balanced[["category", "Detected_Topic", "title"]]

Balanced-sample accuracy (10 articles, 2 per category): 90%



,category,Detected_Topic,title
0,business,Business,Ad sales boost Time Warner profit
1,business,Business,Dollar gains on Greenspan speech
2,entertainment,Entertainment,Gallery unveils interactive tree
3,entertainment,Entertainment,Jarre joins fairytale celebration
4,politics,Politics,Labour plans maternity pay rise
5,politics,Politics,Watchdog probes e-mail deletions
6,sport,Sport,Claxton hunting first major medal
7,sport,Sport,O'Sullivan could run in Worlds
8,tech,Tech,Ink helps drive democracy in Asia
9,tech,Politics,China net cafe culture crackdown


---
# Part 2: Job Postings Analysis — Role Categorization and Requirements Extraction (55 marks)

## Step 1: Load the Dataset

Job postings scraped from job portals, with a `Job Title` and a free-text `Job Description`.

In [22]:
jobs_raw = pd.read_csv("job_title_des.csv")

# The first column is just the original row index from the scrape.
jobs_raw = jobs_raw.rename(columns={"Unnamed: 0": "Job_ID",
                                    "Job Title": "Job_Title",
                                    "Job Description": "Job_Description"})

print("Shape:", jobs_raw.shape)
print("\nMost common titles:")
print(jobs_raw["Job_Title"].value_counts().head(10))
jobs_raw.head()

Shape: (2277, 3)

Most common titles:
Job_Title
JavaScript Developer    166
Java Developer          161
Software Engineer       160
Node js developer       160
iOS Developer           159
PHP Developer           156
Flutter Developer       155
DevOps Engineer         155
Django Developer        152
Machine Learning        152
Name: count, dtype: int64


,Job_ID,Job_Title,Job_Description
0,0,Flutter Developer,We are looking for hire experts flutter developer. So you are eligible this post then ...
1,1,Django Developer,PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ - 04)\nStrong Python experience in API d...
2,2,Machine Learning,"Data Scientist (Contractor)\n\nBangalore, IN\n\nResponsibilities\n\nWe are looking for..."
3,3,iOS Developer,JOB DESCRIPTION:\n\nStrong framework outside of iOS is always a plus\n\niOS experience...
4,4,Full Stack Developer,job responsibility full stack engineer – react role make impact petsmart transforming ...


In [23]:
# Limit the data to the first 25 job postings.
df_jobs = jobs_raw.head(N_JOBS).copy().reset_index(drop=True)
df_jobs["Description_Text"] = df_jobs["Job_Description"].str.strip().str.slice(0, MAX_CHARS)

print("Working set:", df_jobs.shape)
df_jobs[["Job_ID", "Job_Title"]]

Working set: (25, 4)


,Job_ID,Job_Title
0,0,Flutter Developer
1,1,Django Developer
2,2,Machine Learning
3,3,iOS Developer
4,4,Full Stack Developer
5,5,Java Developer
6,6,Full Stack Developer
7,7,JavaScript Developer
8,8,DevOps Engineer
9,9,Software Engineer


## Step 2: Define the Job Category Classification Task (10 marks)

The category list is widened beyond the four in the brief (the brief explicitly allows this) so
that the non-IT postings in this dataset land somewhere sensible, with `Other` as the fallback.
Two few-shot examples of `title + snippet -> category` guide the format.

In [24]:
JOB_CATEGORIES = [
    "Technology/IT",
    "Finance",
    "Marketing",
    "Healthcare",
    "Education",
    "Engineering",
    "Operations/Supply Chain",
    "Human Resources",
    "Sales/Business Development",
    "Other",
]

category_prompt = ChatPromptTemplate([
    ("system",
     "You are a recruitment analyst. Given the following job title and description, categorize the "
     f"job into exactly one of the following domains: {', '.join(JOB_CATEGORIES)}.\n"
     "Judge the domain of the role itself, not the industry of the employer. "
     "If nothing fits, answer 'Other'.\n"
     "Answer with the domain label only - no explanation."),
    # Few-shot example 1
    ("human", "Job Title: Backend Engineer\nDescription: Build and maintain REST APIs in Python, "
              "deploy services on AWS, own CI/CD pipelines."),
    ("ai", "Technology/IT"),
    # Few-shot example 2
    ("human", "Job Title: Accounts Payable Executive\nDescription: Process vendor invoices, perform "
              "monthly reconciliations in Tally and assist with statutory audits."),
    ("ai", "Finance"),
    # Actual posting
    ("human", "Job Title: {title}\nDescription: {description}"),
])


def normalise_category(raw: str) -> str:
    """Map the model's answer onto one of the allowed domains.

    An empty reply raises (so .with_retry() asks again); a real answer that fits no domain becomes
    'Other', the fallback the brief asks for.
    """
    cleaned = raw.strip().strip(".").lower()
    if not cleaned:
        raise ValueError("Empty answer from the model")
    for category in JOB_CATEGORIES:
        if category.lower() in cleaned:
            return category
    # Fall back on a partial match ("Technology" for "Technology/IT") before giving up.
    for category in JOB_CATEGORIES:
        if category.split("/")[0].lower() in cleaned:
            return category
    return "Other"


category_chain = (category_prompt | llm_fast | StrOutputParser() | normalise_category).with_retry(
    stop_after_attempt=RETRY_ATTEMPTS
).with_fallbacks([category_prompt | llm | StrOutputParser() | normalise_category])

In [25]:
# Expected output: show this works for a sample datapoint.
job_sample = df_jobs.iloc[1]

sample_category = category_chain.invoke({
    "title": job_sample["Job_Title"],
    "description": job_sample["Description_Text"],
})

print("JOB TITLE :", job_sample["Job_Title"])
print("CATEGORY  :", sample_category)

JOB TITLE : Django Developer
CATEGORY  : Technology/IT


## Step 3: Define the Requirements Extraction Task (30 marks)

Two approaches are shown, as the brief allows either.

**(a) One sub-prompt per field** — three small chains for skills, education and experience.

In [26]:
skills_prompt = ChatPromptTemplate([
    ("system", "List the key skills or technologies mentioned in the following job description. "
               "Return them as a single comma-separated line, nothing else. "
               "If none are mentioned, return exactly 'Not specified'."),
    ("human", "{description}"),
])

education_prompt = ChatPromptTemplate([
    ("system", "What is the minimum education level required or preferred for this job, if stated? "
               "Answer in a short phrase such as \"Bachelor's degree in Computer Science\". "
               "If the description does not state one, return exactly 'Not specified'."),
    ("human", "{description}"),
])

experience_prompt = ChatPromptTemplate([
    ("system", "What is the minimum years of experience or experience level required for this job, "
               "if mentioned? Answer in a short phrase such as '3+ years' or 'Senior level'. "
               "If the description does not mention one, return exactly 'Not specified'."),
    ("human", "{description}"),
])

skills_chain = (skills_prompt | llm_fast | StrOutputParser()).with_retry(stop_after_attempt=RETRY_ATTEMPTS)
education_chain = (education_prompt | llm_fast | StrOutputParser()).with_retry(stop_after_attempt=RETRY_ATTEMPTS)
experience_chain = (experience_prompt | llm_fast | StrOutputParser()).with_retry(stop_after_attempt=RETRY_ATTEMPTS)

In [27]:
# Expected output: show this works for a sample datapoint.
print("JOB TITLE  :", job_sample["Job_Title"], "\n")
print("SKILLS     :", skills_chain.invoke({"description": job_sample["Description_Text"]}))
print("EDUCATION  :", education_chain.invoke({"description": job_sample["Description_Text"]}))
print("EXPERIENCE :", experience_chain.invoke({"description": job_sample["Description_Text"]}))

JOB TITLE  : Django Developer 



SKILLS     : Python, Django, Flask, REST, RPC, API Frameworks, Linux, SQL, JSON, PyUnit, automated unit testing


EDUCATION  : Not specified


EXPERIENCE : Not specified


**(b) A single composite prompt returning structured JSON** — one call instead of three, and the
Pydantic schema guarantees the three fields come back with the right types. This is the version
used for the full run in Step 4: 3× fewer calls against the rate limit, and the format cannot drift.

In [28]:
class JobRequirements(BaseModel):
    """Requirements and qualifications stated in a job posting."""

    # Every field is required - see the note in Part 1 Step 4: an optional field (one with a
    # default) can be omitted by the model, which silently turns the whole column into defaults.
    Required_Skills: List[str] = Field(
        description=("Specific skills, programming languages, tools or domain knowledge required, "
                     "e.g. ['Python', 'REST APIs', 'CRM software']. Use [] if none are stated."),
    )
    Education_Required: str = Field(
        description=("Minimum education level required or preferred, e.g. \"Bachelor's degree in "
                     "Computer Science\". Use exactly 'Not specified' if the posting states none."),
    )
    Experience_Required: str = Field(
        description=("Years or level of experience required, e.g. '5+ years' or 'Senior level'. "
                     "Use exactly 'Not specified' if the posting mentions none."),
    )


requirements_prompt = ChatPromptTemplate([
    ("system",
     "You are a recruitment analyst. Extract the required skills, education level and years of "
     "experience from the job description below.\n"
     "Extract only what the posting actually states - never infer a requirement that is not written "
     "there. When a field is not mentioned, return 'Not specified' (or an empty list for skills)."),
    ("human", "Job Title: {title}\n\nJob Description:\n{description}"),
])

requirements_chain = (requirements_prompt | structured_output(llm, JobRequirements)).with_retry(
    stop_after_attempt=RETRY_ATTEMPTS
)

In [29]:
# Expected output: show this works for a sample datapoint.
sample_requirements = requirements_chain.invoke({
    "title": job_sample["Job_Title"],
    "description": job_sample["Description_Text"],
})

print("JOB TITLE:", job_sample["Job_Title"], "\n")
print(json.dumps(sample_requirements.model_dump(), indent=2))

JOB TITLE: Django Developer 

{
  "Required_Skills": [
    "Python",
    "API development (REST/RPC)",
    "Django",
    "Flask",
    "Linux",
    "SQL",
    "JSON",
    "PyUnit (unit testing)",
    "verbal and written communication"
  ],
  "Education_Required": "Not specified",
  "Experience_Required": "Not specified"
}


## Step 4: Apply the LLM Chain to Each Job Posting (10 marks)

Per posting: the classification chain gives `Predicted_Category`, and the composite extraction
chain gives the three requirement fields. Both are assigned in one runnable, so each row is a
single `.invoke` and the whole set runs as one `.batch`.

In [30]:
job_chain = RunnablePassthrough.assign(
    Predicted_Category=category_chain,
    Requirements=requirements_chain,
)

start = time.time()

job_inputs = [
    {"title": title, "description": description}
    for title, description in zip(df_jobs["Job_Title"], df_jobs["Description_Text"])
]
job_results = job_chain.batch(job_inputs, config={"max_concurrency": MAX_CONCURRENCY})

print(f"Processed {len(job_results)} job postings in {time.time() - start:.1f}s")

Processed 25 job postings in 193.8s


## Step 5: Update the DataFrame with New Columns (5 marks)

In [31]:
df_jobs["Predicted_Category"] = [r["Predicted_Category"] for r in job_results]

# Required_Skills is kept as a Python list on every row (consistent type, no mixed str/list).
df_jobs["Required_Skills"] = [r["Requirements"].Required_Skills for r in job_results]
df_jobs["Education_Required"] = [r["Requirements"].Education_Required or "Not specified" for r in job_results]
df_jobs["Experience_Required"] = [r["Requirements"].Experience_Required or "Not specified" for r in job_results]

df_jobs_final = df_jobs[
    ["Job_ID", "Job_Title", "Job_Description",
     "Predicted_Category", "Required_Skills", "Education_Required", "Experience_Required"]
]

df_jobs_final[["Job_ID", "Job_Title", "Predicted_Category",
               "Required_Skills", "Education_Required", "Experience_Required"]]

,Job_ID,Job_Title,Predicted_Category,Required_Skills,Education_Required,Experience_Required
0,0,Flutter Developer,Technology/IT,[Flutter],Not specified,1 year (Preferred)
1,1,Django Developer,Technology/IT,"[Python, Django, Flask, REST APIs, RPC, Linux, SQL, JSON, PyUnit, Automated unit testi...",Not specified,Not specified
2,2,Machine Learning,Technology/IT,"[Machine Learning, Deep Learning, Python, Java, statistics, applied mathematics, softw...","Any Graduate or M.Sc. in Computer Science, Mathematics or equivalent, preferably in Ma...",At least 3 years
3,3,iOS Developer,Technology/IT,"[iOS development, Objective-C, Cocoa Touch, Core Data, Core Animation, Core Graphics, ...",Not specified,Not specified
4,4,Full Stack Developer,Technology/IT,"[React, React Native, Redux, Angular, Vue, JavaScript, HTML, CSS, RESTful APIs, HTTP n...",Not specified,"5+ years hands-on web development, 2+ years recent React experience"
5,5,Java Developer,Technology/IT,"[WSDL, SOAP, RESTful, Relational databases, Data access/database queries, Software des...",Not specified,Not specified
6,6,Full Stack Developer,Technology/IT,"[Node.js, Java, MongoDB, Elasticsearch, Redis, React, Angular, Single-page application...","B.Sc degree in Computer Science, Engineering, or a relevant field",Minimum 2 years
7,7,JavaScript Developer,Technology/IT,"[ReactJS, NodeJS, Azure Functions, GraphQL, HTML5, CSS3, JavaScript, REST]","Any graduation, and Any PG and Any Doctorate",3 - 8 years
8,8,DevOps Engineer,Technology/IT,"[Bash, Ruby, Python, Java, Puppet, Chef, Cloudify, CFEngine, Cobbler, Foreman, PHP, Li...",Not specified,Not specified
9,9,Software Engineer,Technology/IT,"[REST API, C/C++ for Linux/Unix, Python, Go]",Not specified,Not specified


In [32]:
# The same record in the JSON shape asked for in the assignment.
job_row = df_jobs_final.iloc[1]

print(json.dumps({
    "Job_Title": job_row["Job_Title"],
    "Job_Description": job_row["Job_Description"][:200].strip() + "... [excerpt]",
    "Predicted_Category": job_row["Predicted_Category"],
    "Required_Skills": job_row["Required_Skills"],
    "Education_Required": job_row["Education_Required"],
    "Experience_Required": job_row["Experience_Required"],
}, indent=2))

{
  "Job_Title": "Django Developer",
  "Job_Description": "PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ - 04)\nStrong Python experience in API development (REST/RPC).\nExperience working with API Frameworks (Django/flask).\nExperience evaluating and improving t... [excerpt]",
  "Predicted_Category": "Technology/IT",
  "Required_Skills": [
    "Python",
    "Django",
    "Flask",
    "REST APIs",
    "RPC",
    "Linux",
    "SQL",
    "JSON",
    "PyUnit",
    "Automated unit testing",
    "Verbal communication",
    "Written communication"
  ],
  "Education_Required": "Not specified",
  "Experience_Required": "Not specified"
}


### Spot-check: does the extraction match what the posting actually says?

In [33]:
for idx in [2, 7, 14]:
    r = df_jobs_final.iloc[idx]
    print("=" * 100)
    print(f"[{idx}] {r['Job_Title']}  ->  {r['Predicted_Category']}")
    print("-" * 100)
    print("DESCRIPTION (first 500 chars):")
    print(r["Job_Description"][:500].strip())
    print("-" * 100)
    print("Skills    :", r["Required_Skills"])
    print("Education :", r["Education_Required"])
    print("Experience:", r["Experience_Required"])
    print()

[2] Machine Learning  ->  Technology/IT
----------------------------------------------------------------------------------------------------
DESCRIPTION (first 500 chars):
Data Scientist (Contractor)

Bangalore, IN

Responsibilities

We are looking for a capable data scientist to join the Analytics team, reporting locally in India Bangalore. This person’s responsibilities include research, design and development of Machine Learning and Deep Learning algorithms to tackle a variety of Fraud oriented challenges. The data scientist will work closely with software engineers and program managers to deliver end-to-end products, including: data collection in big scale and
----------------------------------------------------------------------------------------------------
Skills    : ['Machine Learning', 'Deep Learning', 'Python', 'Java', 'statistics', 'applied mathematics', 'software development']
Education : Any Graduate or M.Sc. in Computer Science, Mathematics or equivalent, preferably in M

In [34]:
# How the 25 postings were distributed, and how often each field was actually stated.
print(df_jobs_final["Predicted_Category"].value_counts(), "\n")

print("Postings with at least one skill listed:",
      (df_jobs_final["Required_Skills"].str.len() > 0).sum(), "/", len(df_jobs_final))
print("Postings stating an education requirement :",
      (df_jobs_final["Education_Required"] != "Not specified").sum(), "/", len(df_jobs_final))
print("Postings stating an experience requirement:",
      (df_jobs_final["Experience_Required"] != "Not specified").sum(), "/", len(df_jobs_final))

Predicted_Category
Technology/IT    25
Name: count, dtype: int64 

Postings with at least one skill listed: 25 / 25
Postings stating an education requirement : 12 / 25
Postings stating an experience requirement: 16 / 25


In [35]:
df_jobs_final.to_csv("part2_job_analysis_output.csv", index=False)
print("Saved -> part2_job_analysis_output.csv")

Saved -> part2_job_analysis_output.csv


---
## Summary

**Part 1 — 30 BBC news articles.** A few-shot `ChatPromptTemplate` classifies each article into one
of the five BBC topics, a second chain writes a 2–3 sentence factual summary, and a third chain —
running *after* the summary, with a Pydantic schema via `with_structured_output` — pulls out
people, organizations and locations. The three are composed with `RunnablePassthrough.assign` into
a single runnable applied to every row, producing `Detected_Topic`, `Summary` and `Key_Entities`
alongside the original columns. Because the dataset carries its own labels, the classification step
is scored rather than just inspected — including on a balanced 2-per-category sample, since
`head(30)` is entirely business articles.

**Part 2 — 25 job postings.** A separate few-shot classification chain assigns a domain, and a
composite structured-output chain returns skills, education and experience in one call. Postings
that simply do not state an education or experience requirement come back as `Not specified`
rather than having one invented — the behaviour the brief asks for, and visible in the spot-check
above.

**Design choices worth noting**

* Two Groq models sized to the task — `gpt-oss-20b` for topic/domain labels and summaries,
  `gpt-oss-120b` for everything that returns structured output. Groq's free tier limits each
  model to 8,000 tokens per minute separately, so the split gives two budgets and roughly halves
  the run time. (20b is not used for extraction: it is unreliable at structured output.)
* Topic classification reads only the headline and lead paragraph — about half the tokens, and
  the balanced check above shows it still classifies correctly.
* An `InMemoryRateLimiter` per model spaces requests evenly, with client retries for the odd 429.
  Retries alone were tried and failed: parallel calls hit the cap together, are all told to retry
  in ~0.2s, and collide again until their retries run out.
* `temperature=0.0` throughout, so classification and extraction are reproducible.
* Structured output (Pydantic schemas) instead of hand-parsing strings, with every field
  **required** — a defaulted field is optional in the schema and can silently come back empty.
* Normaliser functions after each classification chain, so those columns can only hold a label
  from the allowed list.
* A `strip_preamble` step on the summariser, in case the model opens with "Here is a summary".

**Optional bonus (running the full datasets).** The notebook is parameterised: set `N_ARTICLES` and
`N_JOBS` in Section 0 to `len(news_raw)` and `len(jobs_raw)` to process all 2,225 articles and
2,277 postings. On Groq's free tier that is not practical — ~4,500 calls is well past the
1,000-requests-per-day limit, and ~4M tokens is over eight hours at 8,000 tokens per minute. As the
brief advises, a full run should use a local SLM through Ollama instead. Every chain is written
against the `init_chat_model` interface, so that is a one-line change (and drop the rate limiter):

```python
llm = init_chat_model("llama3.2", model_provider="ollama", temperature=0.0)
```